# Missing values experiment: does humidity help?

## Business problem

El Niño and La Niña can disrupt supply chains. The team wants early warning from Pacific buoy measurements. This notebook tests one cleaning choice: whether to retain humidity, which is often missing, when predicting the following month's local sea surface temperature anomaly. This is an exploratory proxy for climate risk, not the team's final ENSO or business-risk label.

## Summary

The raw file has 178,080 daily rows; humidity is missing in 36.9%. After grouping readings by approximate buoy site and month, the experiment used 3,505 training pairs (through 1995) and 1,656 later pairs (1996–1998).

| Model | Holdout rows | Accuracy (fraction predicted correctly) | Macro F1 (average class performance) |
| --- | ---: | ---: | ---: |
| Without humidity | 1,656 | 71.44% | 0.72138 |
| Humidity filled plus missingness flag | 1,656 | 71.44% | 0.72081 |
| Only rows with observed humidity | 1,501 | 71.62% | 0.72332 |

The complete-case row covers a smaller, easier-to-observe subset and cannot be compared directly with the full holdout rows. On the same 1,501 humidity-observed holdout pairs, macro F1 was 0.72337 without humidity, 0.72301 with filled humidity, and 0.72332 with complete-case training. A persistence baseline (predict next month's class to match this month's) scored 0.73931, so the small forests are not yet stronger than a simple forecast.

**Finding:** Omit humidity from this prototype's input features. Filling it did not improve the held-out result, while complete-case training discarded 1,379 training pairs. The difference between the two full-data models is tiny, so this is a provisional cleaning decision for this proxy label. The team should revisit it with its final target and several time-based validation periods.


## Import and inspect the data

Import the libraries used for tables, modeling, and scoring. Run this cell first.


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline


Load the original CSV as a pandas DataFrame. The path works when Jupyter starts in either the notebooks folder or the repository root.


In [5]:
data_path = Path("../data/el_nino_features.csv")
# If Jupyter starts at the repository root, use the second relative path.
if not data_path.exists():
    data_path = Path("data/el_nino_features.csv")
raw = pd.read_csv(data_path)
print("Rows and columns:", raw.shape)
display(raw.head())


Rows and columns: (178080, 11)


,year,month,day,date,latitude,longitude,zon_winds,mer_winds,humidity,air_temp,ss_temp
0,80,3,7,800307,-0.02,-109.46,-6.8,0.7,NaN,26.14,26.24
1,80,3,8,800308,-0.02,-109.46,-4.9,1.1,NaN,25.66,25.97
2,80,3,9,800309,-0.02,-109.46,-4.5,2.2,NaN,25.69,25.28
3,80,3,10,800310,-0.02,-109.46,-3.8,1.9,NaN,25.57,24.31
4,80,3,11,800311,-0.02,-109.46,-4.2,1.5,NaN,25.30,23.19


See which original columns have missing values. The percentages use daily records, before any aggregation.


In [6]:
# isna() marks missing cells; mean() turns those marks into a share of rows.
missing_pct = (raw.isna().mean() * 100).round(1)
display(missing_pct.rename("missing_percent").to_frame())


,missing_percent
year,0.0
month,0.0
day,0.0
date,0.0
latitude,0.0
longitude,0.0
zon_winds,14.1
mer_winds,14.1
humidity,36.9
air_temp,10.2


Check humidity by year. Missingness (the pattern of absent values) matters because older sensors may not have measured humidity at all.


In [7]:
# Years are stored as 80–98; convert them before making time-based splits.
raw["year_full"] = 1900 + raw["year"]
humidity_missing_by_year = raw.groupby("year_full")["humidity"].apply(
    lambda values: values.isna().mean() * 100
)
display(humidity_missing_by_year.round(1).rename("humidity_missing_percent"))


year_full
1980    100.0
1981    100.0
1982    100.0
1983    100.0
1984    100.0
1985    100.0
1986    100.0
1987    100.0
1988    100.0
1989     99.2
1990     91.2
1991     64.0
1992     37.7
1993     23.2
1994     14.5
1995     18.5
1996     13.1
1997     14.9
1998     15.5
Name: humidity_missing_percent, dtype: float64

## Build a next-month label

A site (an approximate buoy location) is represented by latitude and longitude rounded to whole degrees. The original coordinates drift slightly, so grouping by their exact values would split one area into many tiny groups.


In [ ]:
# Rounded coordinates approximate a buoy site despite small location drift.
raw["site_lat"] = raw["latitude"].round().astype(int)
raw["site_lon"] = raw["longitude"].round().astype(int)
site_keys = ["site_lat", "site_lon"]
clim_keys = site_keys + ["month"]
print("Approximate sites:", raw[site_keys].drop_duplicates().shape[0])


Average the daily measurements for each site and month. Each resulting row describes what was known by the end of that month.


In [ ]:
# Count observed SST readings so sparse months can be excluded later.
# One row per approximate site and month lets us forecast the next month.
monthly = (raw.groupby(site_keys + ["year_full", "month"], as_index=False)
           .agg(ss_temp=("ss_temp", "mean"),
                n_sst=("ss_temp", "count"),
                humidity=("humidity", "mean"),
                air_temp=("air_temp", "mean"),
                zon_winds=("zon_winds", "mean"),
                mer_winds=("mer_winds", "mean")))
# Period objects allow an exact consecutive-month check later.
monthly["period"] = pd.PeriodIndex.from_fields(
    year=monthly.year_full, month=monthly.month, freq="M"
)
monthly = monthly.sort_values(site_keys + ["period"]).reset_index(drop=True)
print("Site-month rows:", len(monthly))
display(monthly.head())


Calculate climatology (the usual SST for a site and calendar month). Use only months through 1995, so the later holdout period does not set its own definition of normal.


In [ ]:
# Require at least 10 observed SST values when building the reference average.
reference = monthly.loc[
    (monthly.year_full <= 1995) & (monthly.n_sst >= 10)
]
# This is the preferred normal SST: same site and same calendar month.
site_month_clim = reference.groupby(clim_keys).ss_temp.mean().rename(
    "site_month_clim"
)
# These broader averages cover site/month combinations missing from training.
site_clim = reference.groupby(site_keys).ss_temp.mean().rename("site_clim")
month_clim = reference.groupby("month").ss_temp.mean().rename("month_clim")


Attach the normal SST to each month. When a site has no historical record for that exact calendar month, fall back to its overall average, then the calendar-month average across sites.


In [ ]:
# Each fallback uses the same pre-1996 reference data.
monthly = monthly.join(site_month_clim, on=clim_keys)
monthly = monthly.join(site_clim, on=site_keys)
monthly = monthly.join(month_clim, on="month")
# Fill missing normals in order: exact site/month, site, month, then global.
monthly["clim"] = (monthly.site_month_clim
                   .fillna(monthly.site_clim)
                   .fillna(monthly.month_clim)
                   .fillna(reference.ss_temp.mean()))


Find the current anomaly (the current month's SST minus its normal SST). A positive value means warmer than usual; a negative value means cooler than usual.


In [ ]:
# Only the current month’s anomaly enters the model as a feature.
monthly["current_anomaly"] = monthly.ss_temp - monthly.clim
display(monthly[["year_full", "month", "site_lat", "site_lon",
                 "ss_temp", "clim", "current_anomaly"]].head())


Line up each site's next observed month. The following filter will check that the two calendar months really are consecutive.


In [ ]:
# Shift within each site so next_* columns hold that site’s next record.
grouped = monthly.groupby(site_keys, sort=False)
for column in ["period", "ss_temp", "n_sst", "clim"]:
    monthly["next_" + column] = grouped[column].shift(-1)


Keep consecutive month pairs with at least 10 SST readings in each month. This gives the model a current-month feature row and a reliable next-month answer.


In [ ]:
# The future SST appears only in the label, never in the input columns.
# Reject gaps between months and months with fewer than 10 measured SST values.
usable = monthly.loc[
    (monthly.next_period == monthly.period + 1)
    & (monthly.n_sst >= 10)
    & (monthly.next_n_sst >= 10)
    & monthly.clim.notna()
    & monthly.next_clim.notna()
].copy()
usable["next_anomaly"] = usable.next_ss_temp - usable.next_clim
print("Usable month pairs:", len(usable))


Create the label (the answer a supervised model learns to predict). This is a three-class outcome for the *following month*: cooling below −0.5 °C, normal from −0.5 to +0.5 °C, and warming above +0.5 °C.


In [ ]:
# np.select assigns a class from the future anomaly; it is the target, not an input.
usable["risk_next_month"] = np.select(
    [usable.next_anomaly < -0.5, usable.next_anomaly > 0.5],
    ["cooling", "warming"],
    default="normal"
)
display(usable.risk_next_month.value_counts())


## Separate earlier and later years

Use a time split (train on earlier observations, then evaluate on later ones). This prevents nearby months in a random split from giving an overly optimistic result.


In [ ]:
BASE = ["current_anomaly", "air_temp", "zon_winds", "mer_winds",
        "month", "site_lat", "site_lon"]
CLASSES = ["cooling", "normal", "warming"]
# Later years remain unseen during model fitting and imputer fitting.
train = usable.loc[usable.year_full <= 1995].copy()
holdout = usable.loc[usable.year_full >= 1996].copy()
print("Training pairs:", len(train), "| Holdout pairs:", len(holdout))


Check how much humidity is missing in each period. This matters when judging the model trained only on observed humidity.


In [ ]:
print("Training humidity missing:", f"{train.humidity.isna().mean():.1%}")
print("Holdout humidity missing:", f"{holdout.humidity.isna().mean():.1%}")
print("Observed humidity in holdout:", holdout.humidity.notna().sum())


## Prepare three humidity choices

Imputation (filling missing values) must learn its typical values from training data alone. First calculate humidity medians by approximate site and calendar month, then broader fallbacks.


In [ ]:
# Humidity fill values come from training rows only, never the holdout.
h_site_month = train.groupby(clim_keys).humidity.median().rename("_h_sm")
h_site = train.groupby(site_keys).humidity.median().rename("_h_site")
h_month = train.groupby("month").humidity.median().rename("_h_month")
h_global = train.humidity.median()


Define how to fill humidity in any set of rows. The missingness flag records whether humidity was originally absent; it does not pretend that a filled value was actually measured.


In [ ]:
def fill_humidity(part):
    # Attach typical humidity measured in the training period.
    out = part.copy()
    # Keep the missingness signal before replacing NaN with a median.
    out["humidity_missing"] = out.humidity.isna().astype(int)
    # Join the local median and two fallback medians onto each row.
    out = out.join(h_site_month, on=clim_keys)
    out = out.join(h_site, on=site_keys)
    out = out.join(h_month, on="month")
    # Use broader averages only when a more local average is unavailable.
    out["humidity"] = (out.humidity.fillna(out._h_sm)
                       .fillna(out._h_site)
                       .fillna(out._h_month)
                       .fillna(h_global))
    return out.drop(columns=["_h_sm", "_h_site", "_h_month"])


Apply the same training-based filling rule to the training and holdout rows. The complete-case choice will use only rows whose humidity was observed.


In [ ]:
train_filled = fill_humidity(train)
holdout_filled = fill_humidity(holdout)
observed = holdout.humidity.notna()
print("Filled humidity still missing:", holdout_filled.humidity.isna().sum())


Use one lightweight random forest (a collection of small decision trees) for each choice. Other missing numeric inputs are filled with training medians inside the pipeline (ordered preparation and model steps).


In [ ]:
def fit_predict(train_part, test_part, columns):
    # Fit the imputer on training rows before scoring later rows.
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        # Small trees and a fixed seed keep the comparison lightweight and repeatable.
        RandomForestClassifier(
            n_estimators=80, max_depth=7, min_samples_leaf=8,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42
        )
    )
    # Fit on earlier rows; predict later rows with the same feature columns.
    model.fit(train_part[columns], train_part.risk_next_month)
    return model.predict(test_part[columns])


Train the first model without humidity. This is the full-data baseline for deciding whether humidity adds anything.


In [ ]:
pred_no_humidity = fit_predict(train, holdout, BASE)
print("Predictions made:", len(pred_no_humidity))


Train the second model with filled humidity and its missingness flag. It uses the same training and holdout rows as the first model.


In [ ]:
pred_filled = fit_predict(
    train_filled, holdout_filled, BASE + ["humidity", "humidity_missing"]
)
print("Predictions made:", len(pred_filled))


Train the complete-case model on rows with measured humidity. It can only be checked on holdout rows that also have measured humidity.


In [ ]:
# Complete cases sacrifice rows, and cannot score holdout rows missing humidity.
train_observed = train.loc[train.humidity.notna()]
holdout_observed = holdout.loc[observed]
pred_complete = fit_predict(
    train_observed, holdout_observed, BASE + ["humidity"]
)
print("Training pairs:", len(train_observed),
      "| Scorable holdout pairs:", len(holdout_observed))


## Compare results

Accuracy (share of correct predictions) is intuitive, but macro F1 (the mean of class-level precision/recall balance) gives cooling, normal, and warming equal weight.


In [ ]:
def macro_f1(actual, predicted):
    # Macro averaging gives all three classes equal weight, even if counts differ.
    return f1_score(
        actual, predicted, labels=CLASSES, average="macro", zero_division=0
    )


Compare the two models that can predict every holdout pair. The tiny F1 difference is the direct test of whether filled humidity helped in this experiment.


In [ ]:
# Both rows below use exactly the same 1,656 holdout examples.
full_scores = pd.DataFrame([
    ["No humidity", accuracy_score(holdout.risk_next_month, pred_no_humidity),
     macro_f1(holdout.risk_next_month, pred_no_humidity)],
    ["Filled humidity + missing flag",
     accuracy_score(holdout.risk_next_month, pred_filled),
     macro_f1(holdout.risk_next_month, pred_filled)],
], columns=["model", "accuracy", "macro_f1"])
display(full_scores.round(5))


For a fair three-way comparison, score all models on the *same* 1,501 holdout pairs where humidity was observed. The complete-case model still had fewer training pairs.


In [ ]:
# Restrict all three scores to the same observed-humidity holdout subset.
observed_scores = pd.DataFrame([
    ["No humidity", len(train),
     macro_f1(holdout_observed.risk_next_month,
              pred_no_humidity[observed.to_numpy()])],
    ["Filled humidity + flag", len(train),
     macro_f1(holdout_observed.risk_next_month,
              pred_filled[observed.to_numpy()])],
    ["Complete cases", len(train_observed),
     macro_f1(holdout_observed.risk_next_month, pred_complete)],
], columns=["model", "training_pairs", "macro_f1_on_observed_holdout"])
display(observed_scores.round(5))


Check two simple baselines (reference methods). The majority baseline always predicts the most common training class. Persistence predicts next month's class to match this month's anomaly class.


In [ ]:
# Majority predicts one class for everyone; persistence carries forward current risk.
majority = np.repeat(train.risk_next_month.mode()[0], len(holdout))
persistence = np.select(
    [holdout.current_anomaly < -0.5, holdout.current_anomaly > 0.5],
    ["cooling", "warming"], default="normal"
)
print("Majority macro F1:", round(macro_f1(holdout.risk_next_month, majority), 5))
print("Persistence macro F1:",
      round(macro_f1(holdout.risk_next_month, persistence), 5))
print("Persistence accuracy:",
      round(accuracy_score(holdout.risk_next_month, persistence), 5))


Look at recall (the share of actual cases in a class that the model detected). A single overall score can hide weaker performance on one of the three classes.


In [ ]:
for name, predictions in [
    ("No humidity", pred_no_humidity),
    ("Filled humidity + flag", pred_filled),
]:
    # Recall reveals whether a class is frequently missed despite a decent overall score.
    report = classification_report(
        holdout.risk_next_month, predictions,
        labels=CLASSES, output_dict=True, zero_division=0
    )
    print(name, {label: round(report[label]["recall"], 3)
                 for label in CLASSES})


## Interpretation

The two full-data models both reached 71.44% accuracy. Their macro F1 scores were 0.72138 without humidity and 0.72081 with filled humidity. Thus, this run gives no evidence that humidity improves the next-month proxy forecast. The complete-case score is limited to rows with measured humidity and used 1,379 fewer training pairs.

Persistence scored 0.73931 macro F1 and 73.37% accuracy, beating the forests. This matters: the humidity comparison is useful for cleaning, but the current model is not yet a compelling warning system. The proxy label represents *local monthly SST departure*, not official ENSO events or actual supply-chain disruptions.


## Justification: proposed cleaned modeling dataset

These recommendations concern a future modeling table. The raw CSV remains unchanged.

- [REMOVE] humidity: Omit it from this prototype's predictor columns. It is missing in 36.9% of daily rows, and filling it did not improve accuracy or macro F1 on the full holdout. Keep the raw column available for later experiments with the team's final label.
- [REMOVE] humidity_missing: Omit this flag when humidity itself is excluded. It was created only to accompany the filled-humidity comparison.
- [REMOVE] date and day: Exclude them from the monthly model input table after aggregation; year_full and month identify each time period. Preserve both fields in the raw data for tracing observations.
- [EDIT] zon_winds: Average available daily readings within each site/month. Where the resulting model input is missing, fill with a median learned only from training rows; 14.1% of daily values are missing.
- [EDIT] mer_winds: Use the same monthly averaging and training-median fill as zon_winds; 14.1% of daily values are missing. This experiment kept wind inputs in every model, so it did not measure their individual benefit.
- [EDIT] air_temp: Average available daily readings within each site/month and fill missing model inputs with a training median; 10.2% of daily values are missing.
- [EDIT] ss_temp: Use measured values to calculate monthly SST and its anomaly. Require at least 10 measured SST readings in both the current and following month for a labeled pair. About 9.6% of daily SST values are missing; never fill the following month's SST using information that would not exist at forecast time.
- [EDIT] year: Convert the source's two-digit values (80–98) into year_full (1980–1998) before any date logic or time split.
- [EDIT] latitude and longitude: Round them to approximate site_lat and site_lon for this experiment, because exact buoy coordinates drift. Agree on a better site definition before final training.
- [ADD] current_anomaly: Subtract the training-era normal SST for that site and calendar month from current monthly SST. This gives the model a current signal that accounts for location and season.
- [ADD] risk_next_month: Use the following month's anomaly to create the cooling, normal, or warming training label. Keep this target out of predictor columns; review the ±0.5 °C cutoff and ENSO-risk definition with the team before finalizing the dataset.
